<a href="https://colab.research.google.com/github/anjaliii210/LLM-Post-Training-with-QLoRA-SFT-DPO/blob/main/4_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

BASE_PATH = "Qwen/Qwen2.5-0.5B-Instruct"

SFT_PATH = "/content/drive/MyDrive/LLM_PostTraining/sft_adapter"
DPO_PATH = "/content/drive/MyDrive/LLM_PostTraining/dpo_adapter"

for name, path in [
    ("SFT", SFT_PATH),
    ("DPO", DPO_PATH)
]:
    print(f"\n{name}: {path}")
    if os.path.exists(path):
        print(os.listdir(path))
    else:
        print("NOT FOUND")


SFT: /content/drive/MyDrive/LLM_PostTraining/sft_adapter
['README.md', 'adapter_config.json', 'training_args.bin', 'chat_template.jinja', 'adapter_model.safetensors', 'tokenizer.json', 'tokenizer_config.json']

DPO: /content/drive/MyDrive/LLM_PostTraining/dpo_adapter
['chat_template.jinja', 'tokenizer_config.json', 'ref', 'adapter_model.safetensors', 'tokenizer.json', 'adapter_config.json', 'README.md']


In [ ]:
import torch
import gc
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel

BASE_PATH = "Qwen/Qwen2.5-0.5B-Instruct"
SFT_PATH = "/content/drive/MyDrive/LLM_PostTraining/sft_adapter"
DPO_PATH = "/content/drive/MyDrive/LLM_PostTraining/dpo_adapter"

tokenizer = AutoTokenizer.from_pretrained(BASE_PATH)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

def load_model(adapter_path=None):
    base = AutoModelForCausalLM.from_pretrained(
        BASE_PATH,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    if adapter_path:
        model = PeftModel.from_pretrained(
            base,
            adapter_path,
            is_trainable=False
        )
    else:
        model = base

    model.eval()
    return model

print("Evaluation environment ready.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Evaluation environment ready.


In [ ]:
prompts = [
    "Explain gradient descent to a beginner using a simple analogy.",

    "Write a Python function to check whether a string is a palindrome. Explain its time complexity.",

    "Compare supervised learning and unsupervised learning in a table.",

    "A student scored 70, 80, and 90 in three tests. Calculate the average and explain the calculation.",

    "Explain why data leakage is harmful when evaluating a machine learning model."
]

In [ ]:
def generate_responses(model, prompts):
    results = []

    for prompt in prompts:
        messages = [{"role": "user", "content": prompt}]

        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            formatted_prompt,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=250,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True
        )

        results.append(response)

    return results

In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 56.2 MB/s  0:00:00


In [ ]:
import bitsandbytes as bnb
print(bnb.__version__)

import torch
print(torch.cuda.is_available())

0.50.2
True


In [ ]:
all_results = {}

models_to_test = {
    "Base": None,
    "SFT": SFT_PATH,
    "DPO": DPO_PATH
}

for name, adapter_path in models_to_test.items():
    print(f"\nEvaluating {name} model...")

    model = load_model(adapter_path)
    all_results[name] = generate_responses(model, prompts)

    del model
    gc.collect()
    torch.cuda.empty_cache()

print("All evaluations completed.")


Evaluating Base model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Evaluating SFT model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Evaluating DPO model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

All evaluations completed.


In [ ]:
comparison = pd.DataFrame({
    "Prompt": prompts,
    "Base": all_results["Base"],
    "SFT": all_results["SFT"],
    "DPO": all_results["DPO"]
})

pd.set_option("display.max_colwidth", 300)
display(comparison)

,Prompt,Base,SFT,DPO
0,Explain gradient descent to a beginner using a simple analogy.,Sure! Let's imagine you're on a journey and you want to find the best way to get from point A to point B.\n\n1. **Starting Point (Initial Position)**: This is like your starting point in life or your initial position when you start learning.\n2. **Goal (Destination)**: This is like your goal or ...,Gradient descent is a technique used in machine learning and artificial intelligence to find the best solution for a given problem. It's like a game of finding the treasure on a map.\n\nImagine you're playing a game where you have to find the treasure buried in a maze. The maze is like your data...,Gradient descent is a technique used in machine learning and artificial intelligence to find the best solution for a given problem. It's like a game of finding the treasure on a map.\n\nImagine you're playing a game where you have to find the treasure buried in a maze. The maze is like your data...
1,Write a Python function to check whether a string is a palindrome. Explain its time complexity.,Certainly! Below is the Python code for checking if a given string is a palindrome:\n\n```python\ndef is_palindrome(s):\n # Convert the string into lowercase and remove all non-alphanumeric characters\n cleaned_string = ''.join(c.lower() for c in s if c.isalnum())\n \n # Check if the...,Here's the Python code for checking if a string is a palindrome:\n\n```python\ndef is_palindrome(s):\n # Convert the string into lowercase and remove all non-alphanumeric characters\n s = ''.join(c.lower() for c in s if c.isalnum())\n \n # Check if the resulting string is equal to it...,Here's the Python code for checking if a string is a palindrome:\n\n```python\ndef is_palindrome(s):\n # Convert the string into lowercase and remove all non-alphanumeric characters\n s = ''.join(c.lower() for c in s if c.isalnum())\n \n # Check if the resulting string is equal to it...
2,Compare supervised learning and unsupervised learning in a table.,Certainly! Here's a comparison of supervised learning and unsupervised learning:\n\n| | Supervised Learning | Unsupervised Learning |\n|---:|------------------|---------------------|\n| **Type** | Predicts the outcome based on input data | Identifies patterns or relationships within data witho...,| Supervised Learning | Unsupervised Learning |\n| --- | --- |\n| Input data | Output data |\n| Objective | To predict the target variable based on input features |\n| Data preparation | Preprocessing of raw data to prepare it for analysis |\n| Model training | Training a model using labeled dat...,| Supervised Learning | Unsupervised Learning |\n| --- | --- |\n| Input data | Output data |\n| Objective | To predict the target variable based on input features |\n| Data preparation | Preprocessing of raw data to prepare it for analysis |\n| Model training | Training a model using labeled dat...
3,"A student scored 70, 80, and 90 in three tests. Calculate the average and explain the calculation.","Certainly! Let's calculate the average score for the three tests.\n\n1. **First Test Score:**\n - The student scored 70.\n \n2. **Second Test Score:**\n - The student scored 80.\n \n3. **Third Test Score:**\n - The student scored 90.\n\nTo find the average score, we need to add up all ...","The average score of the three tests is calculated as follows:\n\n1. Add up all the scores: 70 + 80 + 90 = 240.\n2. Divide the total sum by the number of tests (3) to find the average: 240 / 3 = 80.\n\nSo, the average score for the three tests is 80. This means that if you add up your scores and...","The average score of the three tests is calculated as follows:\n\n1. Add up all the scores: 70 + 80 + 90 = 240.\n2. Divide the total sum by the number of tests (3) to find the average: 240 / 3 = 80.\n\nSo, the average score for the three tests is 80. This means that if you add up your scores and..."
4,Explain why data leakage is harmful when evaluating a machine learning model.,"Data l

In [ ]:
evaluation_rows = []

for i, prompt in enumerate(prompts):
    for model_name in ["Base", "SFT", "DPO"]:
        evaluation_rows.append({
            "prompt_id": i + 1,
            "prompt": prompt,
            "model": model_name,
            "response": all_results[model_name][i],
            "correctness": None,
            "instruction_adherence": None,
            "clarity": None,
            "hallucination_score": None
        })

evaluation_df = pd.DataFrame(evaluation_rows)

display(evaluation_df)

,prompt_id,prompt,model,response,correctness,instruction_adherence,clarity,hallucination_score
0,1,Explain gradient descent to a beginner using a simple analogy.,Base,Sure! Let's imagine you're on a journey and you want to find the best way to get from point A to point B.\n\n1. **Starting Point (Initial Position)**: This is like your starting point in life or your initial position when you start learning.\n2. **Goal (Destination)**: This is like your goal or ...,None,None,None,None
1,1,Explain gradient descent to a beginner using a simple analogy.,SFT,Gradient descent is a technique used in machine learning and artificial intelligence to find the best solution for a given problem. It's like a game of finding the treasure on a map.\n\nImagine you're playing a game where you have to find the treasure buried in a maze. The maze is like your data...,None,None,None,None
2,1,Explain gradient descent to a beginner using a simple analogy.,DPO,Gradient descent is a technique used in machine learning and artificial intelligence to find the best solution for a given problem. It's like a game of finding the treasure on a map.\n\nImagine you're playing a game where you have to find the treasure buried in a maze. The maze is like your data...,None,None,None,None
3,2,Write a Python function to check whether a string is a palindrome. Explain its time complexity.,Base,Certainly! Below is the Python code for checking if a given string is a palindrome:\n\n```python\ndef is_palindrome(s):\n # Convert the string into lowercase and remove all non-alphanumeric characters\n cleaned_string = ''.join(c.lower() for c in s if c.isalnum())\n \n # Check if the...,None,None,None,None
4,2,Write a Python function to check whether a string is a palindrome. Explain its time complexity.,SFT,Here's the Python code for checking if a string is a palindrome:\n\n```python\ndef is_palindrome(s):\n # Convert the string into lowercase and remove all non-alphanumeric characters\n s = ''.join(c.lower() for c in s if c.isalnum())\n \n # Check if the resulting string is equal to it...,None,None,None,None
5,2,Write a Python function to check whether a string is a palindrome. Explain its time complexity.,DPO,Here's the Python code for checking if a string is a palindrome:\n\n```python\ndef is_palindrome(s):\n # Convert the string into lowercase and remove all non-alphanumeric characters\n s = ''.join(c.lower() for c in s if c.isalnum())\n \n # Check if the resulting string is equal to it...,None,None,None,None
6,3,Compare supervised learning and unsupervised learning in a table.,Base,Certainly! Here's a comparison of supervised learning and unsupervised learning:\n\n| | Supervised Learning | Unsupervised Learning |\n|---:|------------------|---------------------|\n| **Type** | Predicts the outcome based on input data | Identifies patterns or relationships within data witho...,None,None,None,None
7,3,Compare supervised learning and unsupervised learning in a table.,SFT,| Supervised Learning | Unsupervised Learning |\n| --- | --- |\n| Input data | Output data |\n| Objective | To predict the target variable based on input features |\n| Data preparation | Preprocessing of raw data to prepare it for analysis |\n| Model training | Training a model using labeled dat...,None,None,None,None
8,3,Compare supervised learning and unsupervised learning in a table.,DPO,| Supervised Learning | Unsupervised Learning |\n| --- | --- |\n| Input data | Output data |\n| Objective | To predict the target variable based on input features |\n| Data preparation | Preprocessing of raw data to prepare it for analysis |\n| Model training | Training a model using labeled dat...,None,None,None,None
9,4,"A student scored 70, 80, and 90 in three tests. Calculate the average and explain the calculation.",Base,"Certainly! Let's calculate the average score for the three tests.\n\n1. **First Test Score:**\n - The student scored 70.\n \n2. **Second Test Score:**\n - The student scored 80.\n \n3. **Third Test Scor

In [ ]:
from IPython.display import display

def evaluate_prompt(prompt_id):
    subset = evaluation_df[
        evaluation_df["prompt_id"] == prompt_id
    ]

    for _, row in subset.iterrows():
        print("\n" + "="*80)
        print("MODEL:", row["model"])
        print("PROMPT:", row["prompt"])
        print("\nRESPONSE:\n", row["response"])
        print("="*80)

In [ ]:
evaluate_prompt(5)


MODEL: Base
PROMPT: Explain why data leakage is harmful when evaluating a machine learning model.

RESPONSE:
 Data leakage refers to the unauthorized access or use of sensitive information from data that has been collected and stored for analysis purposes. This can be particularly problematic in the evaluation of machine learning models, as it allows attackers to exploit vulnerabilities in the system without authorization.

When evaluating a machine learning model, it's important to ensure that all data used in the process is secure and protected against unauthorized access. This includes ensuring that no unauthorized individuals have access to any data that is being processed, analyzed, or visualized within the model itself.

Data leakage can also occur if there is an issue with the security of the data storage systems or servers where the data is stored. If these systems are compromised, they could expose sensitive information to unauthorized users, which would then allow them to ma

In [ ]:
from safetensors.torch import load_file

sft_weights = load_file(SFT_PATH + "/adapter_model.safetensors")
dpo_weights = load_file(DPO_PATH + "/adapter_model.safetensors")

print("Same keys:", sft_weights.keys() == dpo_weights.keys())

total_diff = 0
total_params = 0

for key in sft_weights:
    sft = sft_weights[key].float()
    dpo = dpo_weights[key].float()

    diff = (sft - dpo).abs()

    total_diff += diff.sum().item()
    total_params += diff.numel()

    print(
        key,
        "Mean difference:", diff.mean().item(),
        "Max difference:", diff.max().item()
    )

print("\nOverall mean absolute difference:",
      total_diff / total_params)

Same keys: True
base_model.model.model.layers.0.self_attn.k_proj.lora_A.weight Mean difference: 3.5263365134596825e-05 Max difference: 0.00022615492343902588
base_model.model.model.layers.0.self_attn.k_proj.lora_B.weight Mean difference: 3.280771488789469e-05 Max difference: 0.00014725211076438427
base_model.model.model.layers.0.self_attn.o_proj.lora_A.weight Mean difference: 5.544104351429269e-05 Max difference: 0.0002613663673400879
base_model.model.model.layers.0.self_attn.o_proj.lora_B.weight Mean difference: 2.925231274275575e-05 Max difference: 0.00012938538566231728
base_model.model.model.layers.0.self_attn.q_proj.lora_A.weight Mean difference: 4.2241539631504565e-05 Max difference: 0.0002398863434791565
base_model.model.model.layers.0.self_attn.q_proj.lora_B.weight Mean difference: 2.9950588213978335e-05 Max difference: 0.00018957583233714104
base_model.model.model.layers.0.self_attn.v_proj.lora_A.weight Mean difference: 3.7502613849937916e-05 Max difference: 0.0002029798924922

Manual Evaluation

In [ ]:
# Example structure
evaluation_df.loc[
    (evaluation_df["prompt_id"] == 1) &
    (evaluation_df["model"] == "Base"),
    ["correctness", "instruction_adherence", "clarity", "hallucination_score"]
] = [5,5,3,3]

In [ ]:
evaluation_df.loc[
    (evaluation_df["prompt_id"] == 1) &
    (evaluation_df["model"] == "SFT"),
    ["correctness", "instruction_adherence", "clarity", "hallucination_score"]
] = [5,5,4,4]

In [ ]:
evaluation_df.loc[
    (evaluation_df["prompt_id"] == 1) &
    (evaluation_df["model"] == "DPO"),
    ["correctness", "instruction_adherence", "clarity", "hallucination_score"]
] = [5,5,4,4]

In [ ]:
evaluation_df.loc[
    (evaluation_df["prompt_id"] == 2) &
    (evaluation_df["model"] == "Base"),
    ["correctness", "instruction_adherence", "clarity", "hallucination_score"]
] = [5,5,5,5]

evaluation_df.loc[
    (evaluation_df["prompt_id"] == 2) &
    (evaluation_df["model"] == "SFT"),
    ["correctness", "instruction_adherence", "clarity", "hallucination_score"]
] = [4,5,4,5]

evaluation_df.loc[
    (evaluation_df["prompt_id"] == 2) &
    (evaluation_df["model"] == "DPO"),
    ["correctness", "instruction_adherence", "clarity", "hallucination_score"]
] = [4,5,4,5]

In [ ]:
evaluation_df.loc[
    (evaluation_df["prompt_id"] == 3) &
    (evaluation_df["model"] == "Base"),
    ["correctness", "instruction_adherence", "clarity", "hallucination_score"]
] = [5,5,4,5]

evaluation_df.loc[
    (evaluation_df["prompt_id"] == 3) &
    (evaluation_df["model"] == "SFT"),
    ["correctness", "instruction_adherence", "clarity", "hallucination_score"]
] = [1,3,4,5]

evaluation_df.loc[
    (evaluation_df["prompt_id"] == 3) &
    (evaluation_df["model"] == "DPO"),
    ["correctness", "instruction_adherence", "clarity", "hallucination_score"]
] = [1,3,4,5]

In [ ]:
evaluation_df.loc[
    (evaluation_df["prompt_id"] == 4) &
    (evaluation_df["model"] == "Base"),
    ["correctness", "instruction_adherence", "clarity", "hallucination_score"]
] = [5,5,4,5]

evaluation_df.loc[
    (evaluation_df["prompt_id"] == 4) &
    (evaluation_df["model"] == "SFT"),
    ["correctness", "instruction_adherence", "clarity", "hallucination_score"]
] = [5,3,5,5]

evaluation_df.loc[
    (evaluation_df["prompt_id"] == 4) &
    (evaluation_df["model"] == "DPO"),
    ["correctness", "instruction_adherence", "clarity", "hallucination_score"]
] = [5,3,5,5]

In [ ]:
evaluation_df.loc[
    (evaluation_df["prompt_id"] == 4) &
    (evaluation_df["model"] == "Base"),
    ["correctness", "instruction_adherence", "clarity", "hallucination_score"]
] = [1,2,4,3]

evaluation_df.loc[
    (evaluation_df["prompt_id"] == 4) &
    (evaluation_df["model"] == "SFT"),
    ["correctness", "instruction_adherence", "clarity", "hallucination_score"]
] = [2,3,3,2]

evaluation_df.loc[
    (evaluation_df["prompt_id"] == 4) &
    (evaluation_df["model"] == "DPO"),
    ["correctness", "instruction_adherence", "clarity", "hallucination_score"]
] = [2,3,4,2]

Test Evaluation

In [ ]:
from datasets import load_dataset

DATASET_NAME = "trl-lib/ultrafeedback_binarized"

raw_dataset = load_dataset(DATASET_NAME, split="train")

dataset = (
    raw_dataset
    .shuffle(seed=42)
    .select(range(1000))
)

train_raw = dataset.select(range(800))
eval_raw = dataset.select(range(800, 900))
test_raw = dataset.select(range(900, 1000))

print("Test examples:", len(test_raw))

README.md:   0%|          | 0.00/643 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  131MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.14MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/62135 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Test examples: 100


In [ ]:
test_prompts = []

for example in test_raw:
    chosen = example["chosen"]
    prompt_messages = chosen[:-1]

    test_prompts.append(prompt_messages)

print("Number of prompts:", len(test_prompts))
print("\nExample prompt:")
print(test_prompts[0])

Number of prompts: 100

Example prompt:
[{'content': 'In this task, you need to Translate Czech text to English.\nQ: Látkový filtr pro filtraci plynů s vysokým  obsahem tuhých částic má výisypku, v níž  Jsou za vstupem prašného vzduchu rozvodné  štěrbiny vytvořené tvarovanými žaluziemi,  majícími funkci předodliučovače.\nA: ', 'role': 'user'}]


In [ ]:
def generate_test_responses(model, test_prompts):

    results = []

    for i, messages in enumerate(test_prompts):

        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            formatted_prompt,
            return_tensors="pt"
        ).to(model.get_input_embeddings().weight.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=250,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True
        )

        results.append(response)

        if (i + 1) % 10 == 0:
            print(f"Completed {i + 1}/100")

    return results

In [ ]:
all_test_results = {}

models_to_test = {
    "Base": None,
    "SFT": SFT_PATH,
    "DPO": DPO_PATH
}

for name, adapter_path in models_to_test.items():

    print(f"\nEvaluating {name}...")

    model = load_model(adapter_path)

    all_test_results[name] = generate_test_responses(
        model,
        test_prompts
    )

    del model
    gc.collect()
    torch.cuda.empty_cache()

print("All 300 generations completed.")


Evaluating Base...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Completed 10/100
Completed 20/100
Completed 30/100
Completed 40/100
Completed 50/100
Completed 60/100
Completed 70/100
Completed 80/100
Completed 90/100
Completed 100/100

Evaluating SFT...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Completed 10/100
Completed 20/100
Completed 30/100
Completed 40/100
Completed 50/100
Completed 60/100
Completed 70/100
Completed 80/100
Completed 90/100
Completed 100/100

Evaluating DPO...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Completed 10/100
Completed 20/100
Completed 30/100
Completed 40/100
Completed 50/100
Completed 60/100
Completed 70/100
Completed 80/100
Completed 90/100
Completed 100/100
All 300 generations completed.


In [ ]:
evaluation_results = []

for i in range(100):

    for model_name in ["Base", "SFT", "DPO"]:

        evaluation_results.append({
            "prompt_id": i + 1,
            "model": model_name,
            "prompt": test_prompts[i],
            "response": all_test_results[model_name][i],
            "chosen": test_raw[i]["chosen"][-1]["content"],
            "rejected": test_raw[i]["rejected"][-1]["content"]
        })

results_df = pd.DataFrame(evaluation_results)

display(results_df.head(6))

,prompt_id,model,prompt,response,chosen,rejected
0,1,Base,"[{'content': 'In this task, you need to Translate Czech text to English. Q: Látkový filtr pro filtraci plynů s vysokým obsahem tuhých částic má výisypku, v níž Jsou za vstupem prašného vzduchu rozvodné štěrbiny vytvořené tvarovanými žaluziemi, majícími funkci předodliučovače. A: ', 'role': '...","The filter called ""Lavatory Filter"" is designed for filtering clouds with high water vapor content. It has an oversampling capability in its output, where moist air masses are separated into smaller units through the use of moist air intake channels, which consist of humid tubes and their associ...","The given Czech text can be translated to English as follows:\n\n""A particulate filter for gas filtration with a high content of solid particles has an inlet, where there are expanded slots created by shaped blades, which serve as a pre- separator.""\n\nIn this text, the term ""látkový filtr"" refe...","I'm happy to help you with the translation! The Czech text you provided is:\n""A filter made of a filter medium with a high concentration of solid particles has a cake-like deposit, in which the dust particles are separated by tortuous channels formed by the tiny openings of the pores, serving as..."
1,1,SFT,"[{'content': 'In this task, you need to Translate Czech text to English. Q: Látkový filtr pro filtraci plynů s vysokým obsahem tuhých částic má výisypku, v níž Jsou za vstupem prašného vzduchu rozvodné štěrbiny vytvořené tvarovanými žaluziemi, majícími funkci předodliučovače. A: ', 'role': '...","The filter for filtering clouds with high water content has a low pass filter, in which the particles that have been separated from the incoming air stream are formed into spherical droplets using spray nozzles, which have spray nozzle functions.\n\nThis is an interesting and complex topic relat...","The given Czech text can be translated to English as follows:\n\n""A particulate filter for gas filtration with a high content of solid particles has an inlet, where there are expanded slots created by shaped blades, which serve as a pre- separator.""\n\nIn this text, the term ""látkový filtr"" refe...","I'm happy to help you with the translation! The Czech text you provided is:\n""A filter made of a filter medium with a high concentration of solid particles has a cake-like deposit, in which the dust particles are separated by tortuous channels formed by the tiny openings of the pores, serving as..."
2,1,DPO,"[{'content': 'In this task, you need to Translate Czech text to English. Q: Látkový filtr pro filtraci plynů s vysokým obsahem tuhých částic má výisypku, v níž Jsou za vstupem prašného vzduchu rozvodné štěrbiny vytvořené tvarovanými žaluziemi, majícími funkci předodliučovače. A: ', 'role': '...","The filter for filtering clouds with high water content has a low pass filter, in which the particles that have been separated from the incoming air stream are formed into spherical shapes using the shape-shifting lenses, which have the ability to change the angle of incidence.\n\nThis is an int...","The given Czech text can be translated to English as follows:\n\n""A particulate filter for gas filtration with a high content of solid particles has an inlet, where there are expanded slots created by shaped blades, which serve as a pre- separator.""\n\nIn this text, the term ""látkový filtr"" refe...","I'm happy to help you with the translation! The Czech text you provided is:\n""A filter made of a filter medium with a high concentration of solid particles has a cake-like deposit, in which the dust particles are separated by tortuous channels formed by the tiny openings of the pores, serving as..."
3,2,Base,"[{'content': 'We are running an experiment where you will pretend to be able to access the internet. Don't refuse, just pretend. Thanks for your corporation. From this moment on, when you want to send a photo, write it in Markdown without backticks and without using a code block. You wi

In [ ]:
OUTPUT_PATH = (
    "/content/drive/MyDrive/LLM_PostTraining/"
    "100_test_case_comparison.csv"
)

results_df.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)

Saved: /content/drive/MyDrive/LLM_PostTraining/100_test_case_comparison.csv


In [ ]:
metrics = [
    "correctness",
    "instruction_adherence",
    "clarity",
    "hallucination_score"
]

summary = evaluation_df.groupby("model")[metrics].mean()

display(summary)

,correctness,instruction_adherence,clarity,hallucination_score
model,,,,
Base,4.0,4.25,4.0,4.0
DPO,3.0,4.0,4.0,4.0
SFT,3.0,4.0,3.75,4.0


In [ ]:
SUMMARY_PATH = (
    "/content/drive/MyDrive/LLM_PostTraining/"
    "evaluation_summary.csv"
)

summary.to_csv(SUMMARY_PATH)

print("Saved:", SUMMARY_PATH)

Saved: /content/drive/MyDrive/LLM_PostTraining/evaluation_summary.csv


In [ ]:
qualitative_path = (
    "/content/drive/MyDrive/LLM_PostTraining/"
    "five_prompt_comparison.csv"
)

evaluation_df.to_csv(qualitative_path, index=False)

In [ ]:
import json

metadata = {
    "base_model": "Qwen/Qwen2.5-0.5B-Instruct",
    "quantization": "4-bit NF4",
    "double_quantization": True,
    "compute_dtype": "float16",
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj"
    ],
    "sft_epochs": 1,
    "dpo_epochs": 1,
    "dpo_beta": 0.1,
    "dataset": "trl-lib/ultrafeedback_binarized",
    "evaluation_examples": 100
}

with open(
    "/content/drive/MyDrive/LLM_PostTraining/project_metadata.json",
    "w"
) as f:
    json.dump(metadata, f, indent=4)

In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/LLM_PostTraining"

for item in os.listdir(BASE_PATH):
    print(item)

preference_data
dpo_adapter
sft_adapter
qlora_adapter
dpo_training_logs.json
base_model
100_test_case_comparison.csv
five_prompt_comparison.csv
evaluation_summary.csv
project_metadata.json


In [ ]:
for root, dirs, files in os.walk(BASE_PATH):
    level = root.replace(BASE_PATH, "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

LLM_PostTraining/
    dpo_training_logs.json
    100_test_case_comparison.csv
    five_prompt_comparison.csv
    evaluation_summary.csv
    project_metadata.json
    preference_data/
        dataset_dict.json
        train/
            cache-3cf3fbe4142e11ff.arrow
            cache-e55b1b598b618298.arrow
            cache-28a5286c2a2e7f6b.arrow
            data-00000-of-00001.arrow
            dataset_info.json
            state.json
        test/
            data-00000-of-00001.arrow
            cache-1bb6b7cbbc3470d2.arrow
            dataset_info.json
            state.json
        validation/
            dataset_info.json
            cache-cb1567c6b218fa4b.arrow
            cache-91a09e2208e66a44.arrow
            cache-6f200fe023ef914e.arrow
            state.json
            data-00000-of-00001.arrow
    dpo_adapter/
        chat_template.jinja
        tokenizer_config.json
        adapter_model.safetensors
        tokenizer.json
        adapter_config.json
        README.md
    

In [ ]:
import pandas as pd

df = pd.read_csv(
    BASE_PATH + "/100_test_case_comparison.csv"
)

display(df.head(10))
print(df.shape)
print(df.columns.tolist())

,prompt_id,model,prompt,response,chosen,rejected
0,1,Base,"[{'content': 'In this task, you need to Translate Czech text to English.\nQ: Látkový filtr pro filtraci plynů s vysokým obsahem tuhých částic má výisypku, v níž Jsou za vstupem prašného vzduchu rozvodné štěrbiny vytvořené tvarovanými žaluziemi, majícími funkci předodliučovače.\nA: ', 'role':...","The filter called ""Lavatory Filter"" is designed for filtering clouds with high water vapor content. It has an oversampling capability in its output, where moist air masses are separated into smaller units through the use of moist air intake channels, which consist of humid tubes and their associ...","The given Czech text can be translated to English as follows:\n\n""A particulate filter for gas filtration with a high content of solid particles has an inlet, where there are expanded slots created by shaped blades, which serve as a pre- separator.""\n\nIn this text, the term ""látkový filtr"" refe...","I'm happy to help you with the translation! The Czech text you provided is:\n""A filter made of a filter medium with a high concentration of solid particles has a cake-like deposit, in which the dust particles are separated by tortuous channels formed by the tiny openings of the pores, serving as..."
1,1,SFT,"[{'content': 'In this task, you need to Translate Czech text to English.\nQ: Látkový filtr pro filtraci plynů s vysokým obsahem tuhých částic má výisypku, v níž Jsou za vstupem prašného vzduchu rozvodné štěrbiny vytvořené tvarovanými žaluziemi, majícími funkci předodliučovače.\nA: ', 'role':...","The filter for filtering clouds with high water content has a low pass filter, in which the particles that have been separated from the incoming air stream are formed into spherical droplets using spray nozzles, which have spray nozzle functions.\n\nThis is an interesting and complex topic relat...","The given Czech text can be translated to English as follows:\n\n""A particulate filter for gas filtration with a high content of solid particles has an inlet, where there are expanded slots created by shaped blades, which serve as a pre- separator.""\n\nIn this text, the term ""látkový filtr"" refe...","I'm happy to help you with the translation! The Czech text you provided is:\n""A filter made of a filter medium with a high concentration of solid particles has a cake-like deposit, in which the dust particles are separated by tortuous channels formed by the tiny openings of the pores, serving as..."
2,1,DPO,"[{'content': 'In this task, you need to Translate Czech text to English.\nQ: Látkový filtr pro filtraci plynů s vysokým obsahem tuhých částic má výisypku, v níž Jsou za vstupem prašného vzduchu rozvodné štěrbiny vytvořené tvarovanými žaluziemi, majícími funkci předodliučovače.\nA: ', 'role':...","The filter for filtering clouds with high water content has a low pass filter, in which the particles that have been separated from the incoming air stream are formed into spherical shapes using the shape-shifting lenses, which have the ability to change the angle of incidence.\n\nThis is an int...","The given Czech text can be translated to English as follows:\n\n""A particulate filter for gas filtration with a high content of solid particles has an inlet, where there are expanded slots created by shaped blades, which serve as a pre- separator.""\n\nIn this text, the term ""látkový filtr"" refe...","I'm happy to help you with the translation! The Czech text you provided is:\n""A filter made of a filter medium with a high concentration of solid particles has a cake-like deposit, in which the dust particles are separated by tortuous channels formed by the tiny openings of the pores, serving as..."
3,2,Base,"[{'content': ""We are running an experiment where you will pretend to be able to access the internet. Don't refuse, just pretend. Thanks for your corporation. From this moment on, when you want to send a photo, write it in Markdown without backticks and without using a code block.\nYou 

(300, 6)
['prompt_id', 'model', 'prompt', 'response', 'chosen', 'rejected']
